<a href="https://colab.research.google.com/github/IDGS-904-23001532/Flask_DB_IDGS804/blob/main/Copy_of_Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3

# 1. Conectar a la base de datos (se creará el archivo automáticamente)
conn = sqlite3.connect("data_warehouse_solar.db")
cursor = conn.cursor()

# Habilitar el soporte de llaves foráneas en SQLite (esencial para mantener la integridad)
cursor.execute("PRAGMA foreign_keys = ON;")

print("¡Conexión exitosa a SQLite! Creando tablas...")

# =====================================================================
# 2. CREACIÓN DE TABLAS DE DIMENSIONES
# =====================================================================

# Dimensión Tiempo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Tiempo (
    ID_Tiempo INTEGER PRIMARY KEY AUTOINCREMENT,
    Fecha TEXT NOT NULL,
    Hora TEXT NOT NULL,
    Anio INTEGER NOT NULL,
    Mes INTEGER NOT NULL,
    Dia INTEGER NOT NULL,
    Hora_Int INTEGER NOT NULL,
    Minuto INTEGER NOT NULL
);
""")

# Dimensión Dispositivo
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Dispositivo (
    ID_Dispositivo INTEGER PRIMARY KEY AUTOINCREMENT,
    Nombre_Dispositivo TEXT NOT NULL,
    Tipo_Dispositivo TEXT NOT NULL,
    Marca TEXT,
    Modelo TEXT
);
""")

# Dimensión Modo Sistema
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_Mod_Sistema (
    ID_Modo INTEGER PRIMARY KEY AUTOINCREMENT,
    Nombre_Modo TEXT NOT NULL,
    Prioridad TEXT NOT NULL
);
""")

# =====================================================================
# 3. CREACIÓN DE LA TABLA DE HECHOS (Fact_Lecturas_IoT)
# =====================================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fact_Lecturas_IoT (
    ID_Lectura INTEGER PRIMARY KEY AUTOINCREMENT,
    FK_Tiempo INTEGER NOT NULL,
    FK_Dispositivo INTEGER NOT NULL,
    FK_Modo INTEGER NOT NULL,
    Valor_Voltaje REAL,
    Valor_Temperatura REAL,
    Energia_Generada_Wh REAL,
    Energia_Consumida_Wh REAL,
    Porcentaje_Bateria REAL,
    FOREIGN KEY (FK_Tiempo) REFERENCES Dim_Tiempo(ID_Tiempo),
    FOREIGN KEY (FK_Dispositivo) REFERENCES Dim_Dispositivo(ID_Dispositivo),
    FOREIGN KEY (FK_Modo) REFERENCES Dim_Mod_Sistema(ID_Modo)
);
""")

print("Tablas creadas correctamente.")

# =====================================================================
# 4. INSERCIÓN DE DATOS DE PRUEBA (Para validar el modelo)
# =====================================================================

print("Insertando datos de prueba...")

# Insertar en Dim_Mod_Sistema
modos = [
    ("Ahorro de Energía (Eco)", "Baja"),
    ("Carga Rápida (Grid)", "Media"),
    ("Respaldo de Emergencia", "Alta")
]
cursor.executemany("INSERT INTO Dim_Mod_Sistema (Nombre_Modo, Prioridad) VALUES (?, ?);", modos)

# Insertar en Dim_Dispositivo
dispositivos = [
    ("Inversor Central Híbrido", "Inversor", "Growatt", "MIN 5000TL-X"),
    ("Banco de Baterías Litio", "Batería", "Pylontech", "US3000C")
]
cursor.executemany("INSERT INTO Dim_Dispositivo (Nombre_Dispositivo, Tipo_Dispositivo, Marca, Modelo) VALUES (?, ?, ?, ?);", dispositivos)

# Insertar en Dim_Tiempo
tiempos = [
    ("2026-05-23", "12:00:00", 2026, 5, 23, 12, 0),
    ("2026-05-23", "12:01:00", 2026, 5, 23, 12, 1)
]
cursor.executemany("INSERT INTO Dim_Tiempo (Fecha, Hora, Anio, Mes, Dia, Hora_Int, Minuto) VALUES (?, ?, ?, ?, ?, ?, ?);", tiempos)

# Guardar los ID generados para la tabla de hechos
# Nota: En un entorno real, tu script ETL buscaría estos IDs automáticamente.
cursor.execute("INSERT INTO Fact_Lecturas_IoT (FK_Tiempo, FK_Dispositivo, FK_Modo, Valor_Voltaje, Valor_Temperatura, Energia_Generada_Wh, Energia_Consumida_Wh, Porcentaje_Bateria) VALUES (1, 1, 1, 120.5, 35.2, 450.0, 120.0, 85.0);")
cursor.execute("INSERT INTO Fact_Lecturas_IoT (FK_Tiempo, FK_Dispositivo, FK_Modo, Valor_Voltaje, Valor_Temperatura, Energia_Generada_Wh, Energia_Consumida_Wh, Porcentaje_Bateria) VALUES (2, 1, 1, 121.0, 35.6, 465.0, 115.0, 85.5);")

# Confirmar cambios
conn.commit()
print("¡Datos de prueba insertados!")

# =====================================================================
# 5. PRUEBA DE CONSULTA (Simulación de consulta analítica/BI)
# =====================================================================
print("\n--- EJECUTANDO CONSULTA DE PRUEBA (JOIN COMPLETO) ---")
query = """
SELECT
    T.Fecha || ' ' || T.Hora AS Momento,
    D.Nombre_Dispositivo,
    M.Nombre_Modo,
    F.Energia_Generada_Wh,
    F.Porcentaje_Bateria
FROM Fact_Lecturas_IoT F
JOIN Dim_Tiempo T ON F.FK_Tiempo = T.ID_Tiempo
JOIN Dim_Dispositivo D ON F.FK_Dispositivo = D.ID_Dispositivo
JOIN Dim_Mod_Sistema M ON F.FK_Modo = M.ID_Modo;
"""

cursor.execute(query)
resultados = cursor.fetchall()

for fila in resultados:
    print(f"Momento: {fila[0]} | Equipo: {fila[1]} | Modo: {fila[2]} | Gen: {fila[3]}Wh | Bat: {fila[4]}%")

# Cerrar la conexión
conn.close()
print("\nBase de datos cerrada. ¡Todo listo para tu repositorio!")

¡Conexión exitosa a SQLite! Creando tablas...
Tablas creadas correctamente.
Insertando datos de prueba...
¡Datos de prueba insertados!

--- EJECUTANDO CONSULTA DE PRUEBA (JOIN COMPLETO) ---
Momento: 2026-05-23 12:00:00 | Equipo: Inversor Central Híbrido | Modo: Ahorro de Energía (Eco) | Gen: 450.0Wh | Bat: 85.0%
Momento: 2026-05-23 12:01:00 | Equipo: Inversor Central Híbrido | Modo: Ahorro de Energía (Eco) | Gen: 465.0Wh | Bat: 85.5%

Base de datos cerrada. ¡Todo listo para tu repositorio!




```
# This is formatted as code
```

# Tipos y fuentes de datos